# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pratyush457/week-1-/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [3]:
# W06 setup — DuckDB + Hugging Face

%pip -q install duckdb huggingface_hub

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

base = "hf://datasets/FlyRank/internship-warehouse"

march = f"""
read_parquet(
    '{base}/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

april = f"""
read_parquet(
    '{base}/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

print("DuckDB connection ready.")
print("March and April data sources ready.")

DuckDB connection ready.
March and April data sources ready.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
# Section 2: Rebuild decision-time dataset for validation audit

# March decision-time features
march_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        CASE
            WHEN gsc_impressions > 0
            THEN 100.0 * gsc_clicks / gsc_impressions
            ELSE 0
        END AS ctr_pct
    FROM {march}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_impressions > 0
""").df()

# April outcome — target only, never a model feature
future_clicks = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_clicks, 0)) AS future_clicks
    FROM {april}
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

future_clicks["target"] = (
    future_clicks["future_clicks"] > 0
).astype("int8")

# Join March features with April outcome
model_df = march_features.merge(
    future_clicks[
        [
            "client_hash_id",
            "content_hash_id",
            "future_clicks",
            "target"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df["report_date"] = pd.to_datetime(
    model_df["report_date"]
)

print("Rows available:", len(model_df))
print("Positive targets:", int(model_df["target"].sum()))
print("model_df ready.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows available: 3536466
Positive targets: 1617502
model_df ready.


In [5]:
# Section 2: Honest grouped-by-client validation

from sklearn.ensemble import RandomForestClassifier

# ---------------------------------------------------------
# 1. Split clients — same client never appears in both sets
# ---------------------------------------------------------

clients = (
    model_df["client_hash_id"]
    .drop_duplicates()
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

split_point = int(len(clients) * 0.80)

train_clients = set(clients.iloc[:split_point])
valid_clients = set(clients.iloc[split_point:])

train_grouped = model_df[
    model_df["client_hash_id"].isin(train_clients)
].copy()

valid_grouped = model_df[
    model_df["client_hash_id"].isin(valid_clients)
].copy()

print("Grouped training rows:", len(train_grouped))
print("Grouped validation rows:", len(valid_grouped))
print("Training clients:", len(train_clients))
print("Validation clients:", len(valid_clients))

# ---------------------------------------------------------
# 2. Keep training manageable for Colab
# ---------------------------------------------------------

MAX_TRAIN_ROWS = 300_000

if len(train_grouped) > MAX_TRAIN_ROWS:
    train_sample = train_grouped.sample(
        n=MAX_TRAIN_ROWS,
        random_state=42
    )
else:
    train_sample = train_grouped

print("Rows used to train grouped model:", len(train_sample))

# ---------------------------------------------------------
# 3. Decision-time features only
# ---------------------------------------------------------

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr_pct"
]

X_train = train_sample[feature_cols].fillna(0)
y_train = train_sample["target"]

X_valid = valid_grouped[feature_cols].fillna(0)

# ---------------------------------------------------------
# 4. Train same Random Forest configuration as W05
# ---------------------------------------------------------

grouped_model = RandomForestClassifier(
    n_estimators=60,
    max_depth=8,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

grouped_model.fit(X_train, y_train)

# ---------------------------------------------------------
# 5. Model ranking
# ---------------------------------------------------------

valid_grouped["grouped_model_score"] = (
    grouped_model.predict_proba(X_valid)[:, 1]
)

# ---------------------------------------------------------
# 6. Week-4 baseline on SAME validation rows
# ---------------------------------------------------------

valid_grouped["baseline_score"] = (
    np.log1p(valid_grouped["gsc_impressions"])
    * (
        1 -
        valid_grouped["ctr_pct"].clip(
            lower=0,
            upper=100
        ) / 100
    )
)

# ---------------------------------------------------------
# 7. Precision@50
# ---------------------------------------------------------

def precision_at_50(df, score_col):
    top50 = df.nlargest(50, score_col)
    return top50["target"].mean()

grouped_baseline_p50 = precision_at_50(
    valid_grouped,
    "baseline_score"
)

grouped_model_p50 = precision_at_50(
    valid_grouped,
    "grouped_model_score"
)

# ---------------------------------------------------------
# 8. Before vs After
# ---------------------------------------------------------

before_after = pd.DataFrame({
    "validation": [
        "Week-5 time-aware split",
        "Week-6 grouped-by-client split"
    ],
    "model": [
        "Random Forest",
        "Random Forest"
    ],
    "Precision@50": [
        1.000,
        grouped_model_p50
    ]
})

print("\nBEFORE vs AFTER")
display(before_after)

print(
    f"Grouped baseline Precision@50: "
    f"{grouped_baseline_p50:.3f}"
)

print(
    f"Grouped Random Forest Precision@50: "
    f"{grouped_model_p50:.3f}"
)

Grouped training rows: 2372640
Grouped validation rows: 1163826
Training clients: 36
Validation clients: 10
Rows used to train grouped model: 300000

BEFORE vs AFTER


,validation,model,Precision@50
0,Week-5 time-aware split,Random Forest,1.0
1,Week-6 grouped-by-client split,Random Forest,1.0


Grouped baseline Precision@50: 1.000
Grouped Random Forest Precision@50: 1.000


## 3. Leakage audit

The validation audit checks that no future outcome or label-derived field is used as a model feature.

The model inputs are limited to decision-time March signals:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ctr_pct

The April future-click value is used only to construct the evaluation target. It is not included in the feature matrix.

The client identifier is used only to create the grouped validation split and is not used as a model feature.

The audit therefore checks both the explicit feature list and the dataset columns for future or label-derived fields.

In [6]:
# Section 3: Leakage audit

# Features actually used by the model
used_features = feature_cols

# Fields that must never be model inputs
forbidden_features = [
    "future_clicks",
    "target",
    "label",
    "y",
    "next_period_clicks",
    "future_ctr",
    "future_impressions"
]

found_forbidden = [
    col for col in used_features
    if col in forbidden_features
]

# Additional safety check against the feature matrix
feature_matrix_columns = list(X_train.columns)

found_forbidden_in_matrix = [
    col for col in feature_matrix_columns
    if col in forbidden_features
]

print("Model features:", used_features)
print(
    "Forbidden/future features found:",
    sorted(
        set(found_forbidden + found_forbidden_in_matrix)
    )
)

# Client ID is only used for grouping, not modeling
print(
    "Client ID used as model feature:",
    "client_hash_id" in feature_matrix_columns
)

if not found_forbidden and not found_forbidden_in_matrix:
    print("LEAKAGE CHECK: PASSED")
else:
    print("LEAKAGE CHECK: FAILED")

Model features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ctr_pct']
Forbidden/future features found: []
Client ID used as model feature: False
LEAKAGE CHECK: PASSED


## 4. Claim rewrite

### Original claim

The Random Forest achieved a Precision@50 of 1.000 and outperformed the Week-4 baseline, which achieved 0.860, on the Week-5 validation split.

### Rewritten claim

On the evaluated validation data, the Random Forest measured a Precision@50 of 1.000, compared with 0.860 for the Week-4 baseline. Under the stricter grouped-by-client validation, the Random Forest and baseline both measured 1.000 Precision@50. These results show an observed ranking outcome in this validation setup, but they do not establish that the model will perform the same way on other time periods, clients, or future data. The model should therefore be treated as directional decision-support for prioritization and human review, not as a causal or guaranteed predictor of future clicks.

### What I can and cannot claim

**I can claim:** the measured Precision@50 values under the stated validation designs and the observed feature importance and error patterns.

**I cannot claim:** that the model causes higher clicks, proves Google's ranking behavior, or will achieve the same performance on unseen future data outside this validation setup.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before submitting, I confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it.
- [x] The notebook runs top to bottom with no errors (Runtime → Run all).
- [x] No client names, URLs, or private queries anywhere.
- [x] My claims use careful words: observed, measured, directional, decision-support.
- [x] Two research-paper findings were identified with constructive methodology questions.
- [x] My Week-5 model was re-run under a grouped-by-client validation design and compared with the earlier time-aware result.
- [x] The feature audit found no future-window or label-derived inputs in the model features.
- [x] Real validation outcomes were reviewed and the claims were rewritten to match the evidence.
- [x] The notebook is committed to my repo under `work/notebooks/w06_validation_audit.ipynb`.